# 🍐 Little Fig Studio

**Train any LLM on any hardware.** Select your model, configure, launch.

| Feature | Result |
|---|---|
| Quantization | Beats NF4 on 156/156 layers |
| GPU Speed | 7× faster than BnB NF4 |
| CPU Training | 1.1B model in 400MB RAM |
| Optimizer | FigMeZO (−18.6%, original research) |

**Run all cells below ↓**

In [ ]:
# Step 1: Install Little Fig
!git clone https://github.com/ticketguy/littlefig.git /content/littlefig 2>/dev/null || (cd /content/littlefig && git pull)
!pip install -e "/content/littlefig[train]" -q

# Install cloudflared binary
import subprocess, os
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                    '-O', '/usr/local/bin/cloudflared'], check=True)
    os.chmod('/usr/local/bin/cloudflared', 0o755)

# Verify
import sys, importlib
if '/content/littlefig/src' not in sys.path:
    sys.path.insert(0, '/content/littlefig/src')
for key in list(sys.modules.keys()):
    if 'little_fig' in key:
        del sys.modules[key]
importlib.invalidate_caches()

import torch
from little_fig.engine import __version__ as fig_version
print(f'✅ Little Fig v{fig_version}')
print(f'✅ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Launch Little Fig Studio
import subprocess, time, sys
sys.path.insert(0, '/content/littlefig/src')

from pyngrok import ngrok

# Start server
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'little_fig.web.server:app',
     '--host', '0.0.0.0', '--port', '8888'],
    cwd='/content/littlefig/src',
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)

# Tunnel
public_url = ngrok.connect(8888)
print(f'\n🍐 Little Fig Studio is LIVE!')
print(f'\n   👉 {public_url}')
print(f'\n   Pick any model. Configure training. Launch.')
print(f'   Keep this cell running.')


---
## Alternative: Python API (no UI)

Change `MODEL` to any HuggingFace model.

In [ ]:
import sys
sys.path.insert(0, '/content/littlefig/src')
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

model = FigModel.from_pretrained(MODEL, lora_r=16, lora_alpha=32, shared_codebook=True)
print(f'✅ {MODEL} loaded | Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=256,
    batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=5,
    use_packing=True,
)
trainer = FigTrainer(model, config)
trainer.load_dataset('tatsu-lab/alpaca', max_samples=200)
trainer.train()
model.save_adapter('./my_adapter')
print('✅ Training complete! Adapter saved.')

---
*0xticketguy / Harboria Labs | AGPL-3.0*